<a href="https://colab.research.google.com/github/Tomatoboy-893/make-a-AI/blob/main/%E7%B0%A1%E6%98%93AI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

AI作成してみた


gradio インストール(ダウンロードされてない場合のみ)


In [4]:
!pip install gradio tensorflow tensorflow_hub Pillow numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.2/54.2 MB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.1/323.1 kB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.2/95.2 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 85.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.0/72.0 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.5/62.5 kB 4.5 MB/s eta 0:00:00


ライブラリのインポートとAIモデル関連の関数の定義

In [5]:
import gradio as gr
import tensorflow as tf
import tensorflow_hub as hub
import numpy as np
from PIL import Image
import io

# --- グローバル変数としてモデルを定義 ---
detector = None
MODEL_LOADED_SUCCESSFULLY = False

# --- 1. AIモデルのロード関数 ---
def load_ai_model():
    global detector, MODEL_LOADED_SUCCESSFULLY
    if MODEL_LOADED_SUCCESSFULLY: # 既にロード済みなら何もしない
        print("AIモデルは既にロードされています。")
        return

    model_url = "https://tfhub.dev/tensorflow/faster_rcnn/resnet50_v1_640x640/1"
    print("物体検出AIモデルをロードしています... しばらくお待ちください。")
    try:
        detector = hub.load(model_url)
        MODEL_LOADED_SUCCESSFULLY = True
        print("AIモデルのロードが完了しました。")
    except Exception as e:
        print(f"AIモデルのロード中にエラーが発生しました: {e}")
        print("インターネット接続を確認するか、モデルURLが正しいか確認してください。")
        MODEL_LOADED_SUCCESSFULLY = False

# --- 2. 画像をNumPy配列に変換する関数 (今回はGradio用関数に内包) ---
# (参考として残しますが、 predict_with_gradio 関数内で処理します)
# def load_image_into_numpy_array(image_bytes):
#     try:
#         image = Image.open(io.BytesIO(image_bytes))
#     except Exception as e:
#         print(f"画像読み込みエラー: {e}")
#         return None
#     (im_width, im_height) = image.size
#     if image.mode != 'RGB':
#         image = image.convert('RGB')
#     return np.array(image.getdata()).reshape(
#         (im_height, im_width, 3)).astype(np.uint8)

# --- 3. 物体検出と判定を行う関数 ---
def is_person_or_object(image_np, detection_threshold=0.45):
    global detector, MODEL_LOADED_SUCCESSFULLY
    if not MODEL_LOADED_SUCCESSFULLY or detector is None:
        return "エラー: AIモデルがロードされていません。"

    input_tensor = tf.convert_to_tensor(image_np)
    input_tensor = input_tensor[tf.newaxis, ...] # バッチ次元を追加

    try:
        detections = detector(input_tensor)
    except Exception as e:
        return f"推論中にエラーが発生しました: {e}"

    if 'detection_classes' not in detections or 'detection_scores' not in detections:
        return "エラー: モデルの出力形式が予期したものと異なります。"

    detection_classes = detections['detection_classes'][0].numpy().astype(np.int64)
    detection_scores = detections['detection_scores'][0].numpy()

    # COCOデータセットのクラスID 1 は 'person'
    # モデルによっては異なる場合があるので、使用するモデルのドキュメントを確認してください。
    person_class_id = 1

    # 人が検出されたか確認
    for i in range(len(detection_scores)):
        if detection_scores[i] > detection_threshold:
            if detection_classes[i] == person_class_id:
                return "人" # 人が検出された

    # 人が検出されず、何らかの物体が閾値以上で検出された場合
    if np.any(detection_scores > detection_threshold):
        return "物" # 人以外の物体が検出された
    else:
        return "不明 (閾値を超える物体は検出されませんでした)"

print("AIモデル関連の関数の定義が完了しました。")

AIモデル関連の関数の定義が完了しました。


AIモデルのロード実行

In [6]:
# AIモデルをロード（まだロードされていなければ）
if not MODEL_LOADED_SUCCESSFULLY:
    load_ai_model()
else:
    print("AIモデルは既にロード済みです。")

物体検出AIモデルをロードしています... しばらくお待ちください。
AIモデルのロードが完了しました。


Gradioインターフェースの定義とアプリの起動

In [ ]:
# Gradioインターフェース用の予測関数
def predict_with_gradio(image_pil):
    if not MODEL_LOADED_SUCCESSFULLY:
        return "AIモデルがロードされていません。セル3を再実行してモデルをロードしてください。"
    if image_pil is None:
        return "画像が提供されていません。画像をアップロードしてください。"

    # PIL ImageをNumPy配列に変換
    try:
        image_np = np.array(image_pil)
    except Exception as e:
        return f"画像の形式変換中にエラーが発生しました: {e}"

    # アルファチャンネルがある場合(RGBAなど)、RGBに変換 (モデルがRGBを期待する場合)
    if image_np.ndim == 3 and image_np.shape[-1] == 4: # RGBAの場合
        image_np = image_np[..., :3]
    elif image_np.ndim == 2: # グレースケールの場合、RGBに変換
         # GradioのImageコンポーネントはRGBで読み込むので、通常この分岐は不要かもしれません
        image_np = np.stack((image_np,)*3, axis=-1)

    # モデルが3チャンネルのRGB画像を期待していることを確認
    if image_np.ndim != 3 or image_np.shape[-1] != 3:
        return f"画像のチャンネル形式が不正です。RGB画像を指定してください。現在の形式: {image_np.shape}"

    result = is_person_or_object(image_np)
    return result

# Gradioインターフェースの作成
iface = gr.Interface(
    fn=predict_with_gradio,
    inputs=gr.Image(type="pil", label="判定したい画像をアップロード"), # type="pil"でPillow Imageオブジェクトとして受け取る
    outputs=gr.Label(label="判定結果"),
    title="【AIデモ】画像が「人」か「物」か判定します",
    description="AIモデルがアップロードされた画像を解析し、「人」が写っているか、または何らかの「物」が写っているかを判定します。",
    examples=[
        # ここに例として使える画像のURLやローカルパス（Colabの場合はアップロードが必要）をリスト形式で追加できます
        # 例: ["https://example.com/person.jpg", "https://example.com/object.jpg"]
        # Gradio 3.x以降では、サンプルはGradioアプリのUIから直接アップロードする方が簡単かもしれません。
    ],
    allow_flagging='never' # 'never' または 'auto' (ユーザーフィードバック機能)
)

# インターフェースの起動
# share=True にすると、Colab環境から外部アクセス可能な公開URLが生成されます（通常72時間有効）
# debug=True にすると、エラー発生時に詳細情報が表示されて便利です
if MODEL_LOADED_SUCCESSFULLY:
    print("Gradioアプリを起動します...")
    iface.launch(share=True, debug=True)
else:
    print("AIモデルがロードされていないため、Gradioアプリを起動できません。セル3を正常に実行してください。")

/usr/local/lib/python3.11/dist-packages/gradio/interface.py:416: UserWarning: The `allow_flagging` parameter in `Interface` is deprecated.Use `flagging_mode` instead.
  warnings.warn(


Gradioアプリを起動します...
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://56736235635c4ea680.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
